# Haldemann20 H₂O EoS

This section provides comprehensive testing and benchmarking of the `Haldemann20` class implementation,
including validation of the Mazevet et al. (2019) entropy/energy correction.

**Contents:**
1. [Setup and Imports](#setup)
2. [Basic Functionality Tests](#basic-tests)
3. [Mazevet Correction Validation](#mazevet-correction)
4. [Runtime Benchmarking](#runtime)
5. [Diagnostic Plots](#diagnostics)
6. [Edge Cases and Error Handling](#edge-cases)

---

## 1. Setup and Imports <a name="setup"></a>

Import necessary libraries and initialize the EoS instance.
The AQUA P–T table must be available at the path given below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

from paleos.water_eos import (
    Haldemann20, _N_AT_PER_KG, _T_CRIT, _b1, _b2,
    _S0_OLD_PER_ATOM, _S0_NEW_PER_ATOM,
    get_water_phase, get_water_eos, get_water_eos_for_PT,
    _PHASE_MAP,
)

plt.style.use("tableau-colorblind10")

GPa = 1e9
TPa = 1e12

print("✓ Imports successful")

In [ ]:
# Path to the AQUA P-T table (adjust as needed)
TABLE_PATH = '../../AQUA/Tables/aqua_eos_pt_v1_0.dat'

eos = Haldemann20(TABLE_PATH)

print("✓ Haldemann20 EoS instance created")
print(f"  Grid: {eos.n_P} P × {eos.n_T} T points")
print(f"  P range: 10^{eos.log10_P_grid[0]:.1f} – 10^{eos.log10_P_grid[-1]:.1f} Pa")
print(f"  T range: 10^{eos.log10_T_grid[0]:.2f} – 10^{eos.log10_T_grid[-1]:.2f} K")

---
## 2. Basic Functionality Tests <a name="basic-tests"></a>

Test that all public methods execute without errors and return reasonable values
at representative conditions across the H₂O phase diagram.

In [ ]:
methods = [
    ('density', 'kg/m³'),
    ('specific_internal_energy', 'J/kg'),
    ('specific_entropy', 'J/(kg·K)'),
    ('isobaric_heat_capacity', 'J/(kg·K)'),
    ('isochoric_heat_capacity', 'J/(kg·K)'),
    ('thermal_expansion', 'K⁻¹'),
    ('adiabatic_gradient', 'dimensionless'),
]

test_conditions = [
    (1e5,    300,  'Liquid water (1 bar, 300 K)'),
    (1e9,    500,  'Compressed liquid (1 GPa, 500 K)'),
    (10*GPa, 300,  'High-P ice (10 GPa, 300 K)'),
    (50*GPa, 2000, 'Supercritical (50 GPa, 2000 K)'),
    (1*TPa,  5000, 'Deep interior (1 TPa, 5000 K)'),
]

for P, T, desc in test_conditions:
    print(f"\n{desc}:  P = {P:.1e} Pa, T = {T:.0f} K")
    print("=" * 70)
    phase = eos.phase(P, T)
    print(f"  {'phase':30s}: {phase}")

    all_ok = True
    for method_name, units in methods:
        try:
            result = getattr(eos, method_name)(P, T)
            print(f"  {method_name:30s}: {result:12.6e} {units}")
        except Exception as e:
            print(f"  {method_name:30s}: ERROR - {e}")
            all_ok = False

    print("  ✓ OK" if all_ok else "  ✗ Some methods failed")

---
## 3. Mazevet et al. (2019) Correction Validation <a name="mazevet-correction"></a>

The Mazevet et al. (2019) Helmholtz free energy $F_T$ used in AQUA Region 7
requires two corrections, both independent of density:

1. **Sign error in Eq. (13).** The corrected expression differs from the
   erroneous one by
$$F_\mathrm{sign}(T) = 2\, N_\mathrm{at} \bigl[b_1\,\tau\,\ln(1+\tau^{-2}) + b_2\,\tau\,\arctan\tau\bigr]$$

2. **Reference entropy revision.** $S_0$ was revised from $4.9\,k_\mathrm{B}\,n_\mathrm{at}$
   to $9.8\,k_\mathrm{B}\,n_\mathrm{at}$, requiring
$$F_{S_0}(T) = (S_{0,\mathrm{old}} - S_{0,\mathrm{new}})\,N_\mathrm{at}\,T = -4.9\,k_\mathrm{B}\,N_\mathrm{at}\,T$$

The total $F_\mathrm{shift} = F_\mathrm{sign} + F_{S_0}$ is propagated analytically to
entropy ($S_\mathrm{shift} = -\partial F_\mathrm{shift}/\partial T$) and internal energy
($U_\mathrm{shift} = F_\mathrm{shift} + T\,S_\mathrm{shift}$). Note that the $F_{S_0}$ term,
being linear in $T$, contributes a constant entropy shift but cancels exactly
in the internal energy. We verify:
1. Analytical derivatives match numerical finite differences
2. Thermodynamic identity $U = -T^2\,\partial(F/T)/\partial T$ is satisfied
3. Magnitude of the correction across the temperature range

In [ ]:
print("=" * 70)
print("MAZEVET CORRECTION: ANALYTICAL vs NUMERICAL DERIVATIVES")
print("=" * 70)

T_vals = [200, 300, 647, 1000, 3000, 10000, 50000, 100000]

print(f"\n{'T [K]':>8s} {'F_shift [J/kg]':>15s} {'S_shift [J/(kg\u00b7K)]':>20s} "
      f"{'U_shift [J/kg]':>15s} {'dF/dT err':>12s} {'U identity err':>15s}")
print("-" * 95)

all_ok = True
for T in T_vals:
    f_shift = Haldemann20._f_shift(T)
    df_shift = Haldemann20._df_shift_dT(T)
    s_shift = Haldemann20._entropy_shift(T)
    u_shift = Haldemann20._energy_shift(T)

    # Numerical dF/dT check
    dT = T * 1e-6
    df_num = (Haldemann20._f_shift(T + dT) - Haldemann20._f_shift(T - dT)) / (2 * dT)
    df_err = abs(df_shift - df_num) / max(abs(df_shift), 1e-30)

    # Thermodynamic identity: U = -T^2 d(F/T)/dT
    FoT_p = Haldemann20._f_shift(T + dT) / (T + dT)
    FoT_m = Haldemann20._f_shift(T - dT) / (T - dT)
    u_thermo = -T**2 * (FoT_p - FoT_m) / (2 * dT)
    u_err = abs(u_thermo - u_shift) / max(abs(u_shift), 1)

    ok = df_err < 1e-6 and u_err < 1e-6
    if not ok:
        all_ok = False

    print(f"{T:8.0f} {f_shift:15.4e} {s_shift:20.4e} {u_shift:15.4e} "
          f"{df_err:12.2e} {u_err:15.2e} {chr(10003) if ok else chr(10007)}")

print("\n" + (chr(10003) + " All derivative checks passed" if all_ok else chr(10007) + " Some checks failed"))

In [ ]:
# Plot the correction magnitude as a function of temperature
T_range = np.geomspace(150, 1e5, 500)

F_shift = np.array([Haldemann20._f_shift(T) for T in T_range])
S_shift = np.array([Haldemann20._entropy_shift(T) for T in T_range])
U_shift = np.array([Haldemann20._energy_shift(T) for T in T_range])

fig, axes = plt.subplots(1, 3, figsize=(10, 3), dpi=300)

axes[0].semilogx(T_range, F_shift / 1e6)
axes[0].set_xlabel('Temperature [K]')
axes[0].set_ylabel('$F_{\\mathrm{shift}}$ [MJ/kg]')
axes[0].set_title('Free energy shift')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(_T_CRIT, color='gray', ls='--', alpha=0.5, label='$T_\\mathrm{crit}$')
axes[0].legend()

axes[1].semilogx(T_range, S_shift)
axes[1].set_xlabel('Temperature [K]')
axes[1].set_ylabel('$S_{\\mathrm{shift}}$ [J/(kg\u00b7K)]')
axes[1].set_title('Entropy shift')
axes[1].grid(True, alpha=0.3)
axes[1].axvline(_T_CRIT, color='gray', ls='--', alpha=0.5)

axes[2].semilogx(T_range, U_shift / 1e6)
axes[2].set_xlabel('Temperature [K]')
axes[2].set_ylabel('$U_{\\mathrm{shift}}$ [MJ/kg]')
axes[2].set_title('Internal energy shift')
axes[2].grid(True, alpha=0.3)
axes[2].axvline(_T_CRIT, color='gray', ls='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Compare raw (uncorrected) vs corrected entropy and energy at a few conditions
print("=" * 70)
print("CORRECTED vs UNCORRECTED ENTROPY AND ENERGY")
print("=" * 70)

print(f"\n{'P [GPa]':>10s} {'T [K]':>8s} {'S_raw':>14s} {'S_corr':>14s} {'ΔS/S_raw [%]':>14s} "
      f"{'U_raw':>14s} {'U_corr':>14s} {'ΔU/U_raw [%]':>14s}")
print("-" * 100)

check_pts = [
    (10*GPa,  7000),
    (50*GPa,  8000),
    (100*GPa, 9000),
    (1*TPa,  10000),
    (10*TPa, 30000),
]

for P, T in check_pts:
    s_raw = eos._raw_entropy(P, T)
    s_corr = eos.specific_entropy(P, T)
    u_raw = eos._raw_internal_energy(P, T)
    u_corr = eos.specific_internal_energy(P, T)

    ds_pct = 100 * (s_corr - s_raw) / abs(s_raw) if abs(s_raw) > 1e-10 else np.nan
    du_pct = 100 * (u_corr - u_raw) / abs(u_raw) if abs(u_raw) > 1e-10 else np.nan

    print(f"{P/GPa:10.2f} {T:8.0f} {s_raw:14.4e} {s_corr:14.4e} {ds_pct:14.2f} "
          f"{u_raw:14.4e} {u_corr:14.4e} {du_pct:14.2f}")

In [ ]:
# Relative change in entropy and internal energy in the supercritical phase
# (AQUA Region 7 — the only region affected by the Mazevet+19 correction)
print("=" * 70)
print("RELATIVE CHANGE IN S AND U (SUPERCRITICAL PHASE ONLY)")
print("=" * 70)

P_sample = np.geomspace(10**eos.log10_P_grid[0], 10**eos.log10_P_grid[-1], 150)
T_sample = np.geomspace(10**eos.log10_T_grid[0], 10**eos.log10_T_grid[-1], 150)

ds_rel = []  # |ΔS| / |S_raw|
du_rel = []  # |ΔU| / |U_raw|
n_super = 0
n_total = 0

for P in P_sample:
    for T in T_sample:
        n_total += 1
        if eos.phase(P, T) != "supercritical":
            continue
        n_super += 1

        s_raw = eos._raw_entropy(P, T)
        u_raw = eos._raw_internal_energy(P, T)
        s_shift = Haldemann20._entropy_shift(T)
        u_shift = Haldemann20._energy_shift(T)

        if abs(s_raw) > 1e-10:
            ds_rel.append(abs(s_shift / s_raw))
        if abs(u_raw) > 1e-10:
            du_rel.append(abs(u_shift / u_raw))

ds_rel = np.array(ds_rel)
du_rel = np.array(du_rel)

print(f"
Grid: {len(P_sample)} × {len(T_sample)} = {n_total} points, "
      f"{n_super} in supercritical phase ({100*n_super/n_total:.1f}%)")
print(f"  P range: {P_sample[0]:.1e} – {P_sample[-1]:.1e} Pa")
print(f"  T range: {T_sample[0]:.1f} – {T_sample[-1]:.1f} K")

print(f"
{'':20s} {'median':>10s} {'mean':>10s} {'max':>10s} {'p99':>10s}")
print("-" * 62)
print(f"{'|ΔS| / |S_raw|':20s} {np.median(ds_rel):10.2e} {np.mean(ds_rel):10.2e} "
      f"{np.max(ds_rel):10.2e} {np.percentile(ds_rel, 99):10.2e}")
print(f"{'|ΔU| / |U_raw|':20s} {np.median(du_rel):10.2e} {np.mean(du_rel):10.2e} "
      f"{np.max(du_rel):10.2e} {np.percentile(du_rel, 99):10.2e}")


---
## 4. Runtime Benchmarking <a name="runtime"></a>

Measure execution time for scalar and vectorized calls.

In [ ]:
# Scalar call timing
n_calls = 1000
P_bench = 50 * GPa
T_bench = 3000

print(f"Runtime benchmark: {n_calls} scalar calls at P = {P_bench/GPa:.0f} GPa, T = {T_bench:.0f} K")
print("=" * 70)

for method_name, units in methods:
    method = getattr(eos, method_name)
    _ = method(P_bench, T_bench)  # warm-up

    start = time.perf_counter()
    for _ in range(n_calls):
        method(P_bench, T_bench)
    elapsed = time.perf_counter() - start

    print(f"  {method_name:30s}: {elapsed/n_calls*1e3:8.4f} ms/call")

print("\n✓ Runtime benchmark complete")

---
## 5. Diagnostic Plots <a name="diagnostics"></a>

Visualize key thermodynamic quantities across the P-T range relevant to planetary interiors.

In [ ]:
# Density vs Pressure along isotherms
print("=" * 70)
print("DIAGNOSTIC PLOT: Density vs Pressure")
print("=" * 70)

P_range = np.geomspace(1e5, 100*TPa, 200)
T_isotherms = [300, 1000, 3000, 10000, 30000, 100000]

fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

for T in T_isotherms:
    rho_vals = []
    P_ok = []
    for P in P_range:
        try:
            rho_vals.append(eos.density(P, T))
            P_ok.append(P)
        except Exception:
            pass
    if len(P_ok) > 0:
        ax.loglog(np.array(P_ok) / GPa, rho_vals, label=f'$T = {T}$ K')

ax.set_xlabel('Pressure [GPa]')
ax.set_ylabel('Density [kg/m³]')
ax.set_title('H₂O Density Isotherms (AQUA)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

In [ ]:
# Adiabatic gradient: tabulated vs from corrected entropy
print("=" * 70)
print("DIAGNOSTIC PLOT: Adiabatic Gradient — Table vs Corrected Entropy")
print("=" * 70)

P_test_ad = np.geomspace(10*GPa, 100*TPa, 60)
T_test_ad = [7000, 10000, 20000]

fig, axes = plt.subplots(1, len(T_test_ad), figsize=(12, 3), dpi=300, sharey=True)

for ax, T in zip(axes, T_test_ad):
    nad_tab = []
    nad_corr = []
    P_ok = []

    for P in P_test_ad:
        try:
            nad_tab.append(eos.adiabatic_gradient(P, T))
            nad_corr.append(eos.adiabatic_gradient_from_corrected_entropy(P, T))
            P_ok.append(P)
        except Exception:
            pass

    if len(P_ok) > 0:
        ax.semilogx(np.array(P_ok) / GPa, nad_tab, '-', label='Table $\\nabla_{\\mathrm{ad}}$')
        ax.semilogx(np.array(P_ok) / GPa, nad_corr, '--', label='From corrected $S$')

    ax.set_xlabel('Pressure [GPa]')
    ax.set_title(f'$T = {T}$ K')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('$\\nabla_{\\mathrm{ad}}$')
plt.tight_layout()
plt.show()

In [ ]:
# Speed of sound vs Pressure
print("=" * 70)
print("DIAGNOSTIC PLOT: Speed of Sound")
print("=" * 70)

fig, ax = plt.subplots(figsize=(8, 5), dpi=300)

for T in [300, 1000, 3000, 10000]:
    w_vals = []
    P_ok = []
    for P in P_range:
        try:
            w_vals.append(eos.speed_of_sound(P, T))
            P_ok.append(P)
        except Exception:
            pass
    if P_ok:
        ax.loglog(np.array(P_ok)/GPa, w_vals, label=f'$T = {T}$ K')

ax.set_xlabel('Pressure [GPa]')
ax.set_ylabel('Speed of sound [m/s]')
ax.set_title('H₂O Bulk Speed of Sound (AQUA)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

---
## 6. Edge Cases and Error Handling <a name="edge-cases"></a>

Test behavior at the grid boundaries and with out-of-range queries.

In [ ]:
print("=" * 70)
print("EDGE CASES AND ERROR HANDLING")
print("=" * 70)

# Test 1: Grid boundary conditions
print("\n1. Near grid boundaries:")
print("-" * 50)

P_min = 10**eos.log10_P_grid[0]
P_max = 10**eos.log10_P_grid[-1]
T_min = 10**eos.log10_T_grid[0]
T_max = 10**eos.log10_T_grid[-1]

boundary_pts = [
    (P_min * 1.01, T_min * 1.01, 'Near low-P, low-T corner'),
    (P_max * 0.99, T_max * 0.99, 'Near high-P, high-T corner'),
    (P_min * 1.01, T_max * 0.99, 'Near low-P, high-T corner'),
    (P_max * 0.99, T_min * 1.01, 'Near high-P, low-T corner'),
]

for P, T, desc in boundary_pts:
    try:
        rho = eos.density(P, T)
        print(f"  {desc}: ρ = {rho:.2e} kg/m³ ✓")
    except Exception as e:
        print(f"  {desc}: {e}")

# Test 2: Out of range
print("\n2. Out-of-range queries (should raise):")
print("-" * 50)

oor_pts = [
    (P_min * 0.1, 300, 'Below P_min'),
    (1e5, T_min * 0.1, 'Below T_min'),
]

for P, T, desc in oor_pts:
    try:
        eos.density(P, T)
        print(f"  {desc}: ✗ No error raised!")
    except Exception as e:
        print(f"  {desc}: ✓ Raised {type(e).__name__}")

# H₂O Phase Diagram

This section tests phase determination, generates property maps across the full P-T space,
and computes the thermodynamic consistency parameter Δ (Eq. 29 of Haldemann et al. 2020).

**Contents:**
1. [Setup](#setup-phase)
2. [Phase Diagram Visualization](#phase-diagram)
3. [Phase Determination Tests](#phase-tests)
4. [EoS Selection Integration](#eos-selection)
5. [Thermodynamic Properties Maps](#thermo-maps)
6. [Thermodynamic Consistency](#delta)

---

## Setup <a name="setup-phase"></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

from paleos.water_eos import (
    Haldemann20,
    get_water_phase, get_water_eos, get_water_eos_for_PT,
    _PHASE_MAP,
)

plt.style.use('tableau-colorblind10')

GPa = 1e9
TPa = 1e12

TABLE_PATH = '../../AQUA/Tables/aqua_eos_pt_v1_0.dat'
eos = Haldemann20(TABLE_PATH)

print("✓ Setup complete")

---
## Phase Diagram Visualization <a name="phase-diagram"></a>

Generate the H₂O phase diagram from the AQUA phase-ID grid, reproducing Fig. 2
of Haldemann et al. (2020).

In [ ]:
# Phase map on the AQUA P-T grid
print("Generating phase map on P-T grid...")

P_grid = np.geomspace(1e-1, 100*TPa, 200)       # 1 μbar to 100 TPa
T_grid = np.geomspace(100, 1e5, 200)            # 100 K to 100 000 K
TT, PP = np.meshgrid(T_grid, P_grid)

phase_grid = np.empty(PP.shape, dtype=object)
for i in range(PP.shape[0]):
    for j in range(PP.shape[1]):
        try:
            phase_grid[i, j] = eos.phase(PP[i, j], TT[i, j])
        except Exception:
            phase_grid[i, j] = 'unknown'

# Assign integer codes for plotting
all_phases = sorted(set(phase_grid.ravel()))
phase_to_int = {p: i for i, p in enumerate(all_phases)}
phase_int = np.vectorize(lambda p: phase_to_int.get(p, -1))(phase_grid)

fig, ax = plt.subplots(figsize=(9, 7), dpi=300)
cmap = plt.cm.get_cmap('tab10', len(all_phases))
pcm = ax.pcolormesh(TT, PP / GPa, phase_int, cmap=cmap, shading='auto')

# Colorbar with phase labels
cbar = fig.colorbar(pcm, ax=ax, ticks=range(len(all_phases)))
cbar.ax.set_yticklabels(all_phases, fontsize=8)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Temperature [K]', fontsize=12)
ax.set_ylabel('Pressure [GPa]', fontsize=12)
ax.set_title('H₂O Phase Diagram (AQUA)', fontsize=13)
ax.grid(True, alpha=0.2, which='both')

plt.tight_layout()
plt.show()

# Phase statistics
print(f"\nPhase distribution ({PP.size} grid points):")
for p in all_phases:
    count = np.sum(phase_grid == p)
    print(f"  {p:20s}: {count:6d} ({100*count/PP.size:.1f}%)")

---
## Phase Determination Tests <a name="phase-tests"></a>

Test `get_water_phase()` at representative conditions.

In [ ]:
print("Testing get_water_phase() at representative conditions:")
print("=" * 80)

test_cases = [
    (1e-4, 250,  'solid-ice-Ih',  'Ice Ih at very low P'),
    (1e-4, 300,  'liquid',        'Liquid at 1 bar, 300 K'),
    (1e-4, 500,  'vapor',         'Steam at 1 bar'),
    (5,    300,  'solid-ice-VII', 'Ice-VII at 5 GPa'),
    (100,  300,  'solid-ice-X',   'Ice-X at 100 GPa'),
    (10,   2000, 'supercritical', 'Supercritical at 10 GPa'),
    (100,  5000, 'supercritical', 'Deep interior'),
]

print(f"{'P (GPa)':>10s} {'T (K)':>8s} {'Expected':>18s} {'Got':>18s} {'Status':>8s}")
print("-" * 68)

all_passed = True
for P_gpa, T, expected, desc in test_cases:
    P = P_gpa * GPa
    got = get_water_phase(P, T, eos=eos)
    ok = got == expected
    if not ok:
        all_passed = False
    print(f"{P_gpa:10.4f} {T:8.0f} {expected:>18s} {got:>18s} {'✓' if ok else '✗':>8s}")

print("\n" + ("✓ All phase tests passed" if all_passed else "✗ Some phase tests failed"))

---
## EoS Selection Integration <a name="eos-selection"></a>

Test `get_water_eos_for_PT()` returns valid (eos, phase) tuples.

In [ ]:
print("Testing get_water_eos_for_PT():")
print("=" * 70)

integration_pts = [
    (1e5,    300),
    (10*GPa, 300),
    (50*GPa, 2000),
    (1*TPa,  10000),
    (100*TPa,50000),
]

for P, T in integration_pts:
    eos_inst, phase = get_water_eos_for_PT(P, T, eos=eos)
    rho = eos_inst.density(P, T)
    print(f"  P = {P/GPa:10.2f} GPa, T = {T:7.0f} K  →  {phase:18s}  ρ = {rho:.2f} kg/m³")

print("\n✓ Integration test complete")

---
## Thermodynamic Properties Maps <a name="thermo-maps"></a>

Generate and visualize all thermodynamic quantities across the P-T space.

In [ ]:
# Generate thermodynamic properties on P-T grid
print("Generating thermodynamic properties on P-T grid...")
print("=" * 80)

P_grid = np.geomspace(1e-1, 100*TPa, 150)       # 1 μbar to 100 TPa
T_grid = np.geomspace(100, 1e5, 150)            # 100 K to 100 000 K
TT, PP = np.meshgrid(T_grid, P_grid)

properties = [
    ('density', 'kg/m³'),
    ('specific_internal_energy', 'J/kg'),
    ('specific_entropy', 'J/(kg·K)'),
    ('isobaric_heat_capacity', 'J/(kg·K)'),
    ('isochoric_heat_capacity', 'J/(kg·K)'),
    ('thermal_expansion', 'K⁻¹'),
    ('adiabatic_gradient', ''),
]

prop_grids = {name: np.full_like(PP, np.nan) for name, _ in properties}
phase_grid_str = np.empty(PP.shape, dtype=object)
n_failures = 0

for i in range(PP.shape[0]):
    for j in range(PP.shape[1]):
        P, T = PP[i, j], TT[i, j]
        try:
            phase_grid_str[i, j] = eos.phase(P, T)
        except Exception:
            phase_grid_str[i, j] = 'unknown'

        for prop_name, _ in properties:
            try:
                prop_grids[prop_name][i, j] = getattr(eos, prop_name)(P, T)
            except Exception:
                n_failures += 1

print(f"✓ Grid computed: {PP.shape[0]} × {PP.shape[1]} = {PP.size:,} points")
print(f"  Non-convergences: {n_failures} ({100*n_failures/PP.size/len(properties):.2f}%)")

In [ ]:
# Helper function to plot a property map
def plot_property_map(prop_name, units, prop_grid, vmin=None, vmax=None, log_scale=True):
    fig, ax = plt.subplots(figsize=(10, 7), dpi=300)

    if log_scale:
        data = np.log10(np.abs(prop_grid))
        label = f'log₁₀ |{prop_name}|' + (f' [{units}]' if units else '')
    else:
        data = prop_grid
        label = prop_name + (f' [{units}]' if units else '')

    pcm = ax.pcolormesh(TT, PP / GPa, data, shading='auto', cmap='viridis',
                         vmin=vmin, vmax=vmax)
    fig.colorbar(pcm, ax=ax, label=label)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Temperature [K]', fontsize=12)
    ax.set_ylabel('Pressure [GPa]', fontsize=12)
    ax.grid(True, alpha=0.15, which='both')

    plt.tight_layout()
    plt.show()

In [ ]:
plot_property_map('$\\rho$', 'kg/m³', prop_grids['density'], log_scale=True)

In [ ]:
plot_property_map('$U$', 'J/kg', prop_grids['specific_internal_energy'], log_scale=True, vmin=6.5, vmax=9)

In [ ]:
plot_property_map('$S$', 'J/(kg·K)', prop_grids['specific_entropy'], log_scale=True, vmin=2)

In [ ]:
plot_property_map('$C_P$', 'J/(kg·K)', prop_grids['isobaric_heat_capacity'], log_scale=True, vmin=2.5, vmax=4.5)

In [ ]:
plot_property_map('$C_V$', 'J/(kg·K)', prop_grids['isochoric_heat_capacity'], log_scale=True, vmin=2.5, vmax=4.5)

In [ ]:
plot_property_map('$\\alpha$', 'K⁻¹', prop_grids['thermal_expansion'], log_scale=True)

In [ ]:
plot_property_map('$\\nabla_\\mathrm{ad}$', '', prop_grids['adiabatic_gradient'], log_scale=False)

---
## Thermodynamic Consistency Δ <a name="delta"></a>

Compute the thermodynamic consistency parameter from Haldemann et al. (2020) Eq. 29:

$$\Delta_{\mathrm{Th.c.}} \equiv 1 - \frac{\rho^2\,(\partial S/\partial P)_T}{(\partial\rho/\partial T)_P}$$

For a thermodynamically consistent EoS derived from a single free energy potential, Δ = 0.
Deviations indicate interpolation artifacts at region boundaries or near phase transitions.

In [ ]:
# Compute Delta on the P-T grid
print("Computing thermodynamic consistency parameter Δ...")

Delta = np.full_like(PP, np.nan)

dP_rel = 1e-4
dT_rel = 1e-4

for i in range(PP.shape[0]):
    for j in range(PP.shape[1]):
        P, T = PP[i, j], TT[i, j]

        try:
            dP = P * dP_rel
            dT = T * dT_rel

            # (dS/dP)_T — using corrected entropy
            S_plus = eos.specific_entropy(P + dP, T)
            S_minus = eos.specific_entropy(P - dP, T)
            dS_dP = (S_plus - S_minus) / (2 * dP)

            # (drho/dT)_P
            rho_plus = eos.density(P, T + dT)
            rho_minus = eos.density(P, T - dT)
            drho_dT = (rho_plus - rho_minus) / (2 * dT)

            rho = eos.density(P, T)

            if abs(drho_dT) > 1e-20:
                Delta[i, j] = 1 - rho**2 * dS_dP / drho_dT

        except Exception:
            continue

print("✓ Done")

In [ ]:
plot_property_map('$\\Delta_{\\mathrm{Th.c.}}$', '', Delta, log_scale=True, vmin=-6, vmax=0)

In [ ]:
# Statistics on thermodynamic consistency parameter Delta
print("=" * 70)
print("Thermodynamic Consistency: Δ = 1 - ρ² (∂S/∂P)_T / (∂ρ/∂T)_P")
print("Perfect consistency: Δ = 0")
print("=" * 70)

valid_mask = np.isfinite(Delta)
Delta_valid = Delta[valid_mask]

print(f"\nTotal grid points:    {Delta.size:>10}")
print(f"Valid points:         {valid_mask.sum():>10} ({100*valid_mask.sum()/Delta.size:.1f}%)")
print(f"Invalid/failed:       {(~valid_mask).sum():>10}")

print(f"\n{'Statistic':<20} {'Value':>15} {'|Δ|':>15}")
print("-" * 52)
print(f"{'Mean':<20} {np.mean(Delta_valid):>15.2e} {np.mean(np.abs(Delta_valid)):>15.2e}")
print(f"{'Median':<20} {np.median(Delta_valid):>15.2e} {np.median(np.abs(Delta_valid)):>15.2e}")
print(f"{'Std':<20} {np.std(Delta_valid):>15.2e}")
print(f"{'Min':<20} {np.min(Delta_valid):>15.2e}")
print(f"{'Max':<20} {np.max(Delta_valid):>15.2e}")

# Percentiles
print(f"\n{'Percentiles of |Δ|':^52}")
print("-" * 52)
abs_Delta = np.abs(Delta_valid)
for p in [50, 90, 95, 99, 99.9]:
    val = np.percentile(abs_Delta, p)
    print(f"  {p:>5.1f}%:  |Δ| ≤ {val:.2e}")

# Consistency thresholds
print(f"\n{'Consistency Assessment':^52}")
print("-" * 52)
for thresh in [1e-10, 1e-8, 1e-6, 1e-4, 1e-2, 0.1, 1.0]:
    count = np.sum(abs_Delta <= thresh)
    pct = 100 * count / len(abs_Delta)
    print(f"  |Δ| ≤ {thresh:<8.0e}:  {count:>8} points ({pct:>6.2f}%)")

In [ ]:
# Per-phase statistics
print(f"\n{'Statistics by Phase':^70}")
print("-" * 70)

all_phase_labels = sorted(set(phase_grid_str.ravel()))
print(f"{'Phase':<20} {'Count':>8} {'Mean |Δ|':>12} {'Median |Δ|':>12} {'Max |Δ|':>12}")
print("-" * 68)

for phase in all_phase_labels:
    mask = (phase_grid_str == phase) & valid_mask
    if mask.sum() > 0:
        abs_D = np.abs(Delta[mask])
        print(f"{phase:<20} {mask.sum():>8} {np.mean(abs_D):>12.2e} "
              f"{np.median(abs_D):>12.2e} {np.max(abs_D):>12.2e}")
    else:
        print(f"{phase:<20} {0:>8} {'N/A':>12} {'N/A':>12} {'N/A':>12}")

# Flag problematic regions
print(f"\n{'Problematic Regions (|Δ| > 0.1)':^70}")
print("-" * 70)
problem_mask = np.abs(Delta) > 0.1
if problem_mask.sum() > 0:
    P_prob = PP[problem_mask]
    T_prob = TT[problem_mask]
    phase_prob = phase_grid_str[problem_mask]

    for phase in all_phase_labels:
        pmask = phase_prob == phase
        if pmask.sum() > 0:
            P_range = (P_prob[pmask].min()/GPa, P_prob[pmask].max()/GPa)
            T_range = (T_prob[pmask].min(), T_prob[pmask].max())
            print(f"  {phase}: {pmask.sum()} points")
            print(f"    P: {P_range[0]:.1f} – {P_range[1]:.1f} GPa")
            print(f"    T: {T_range[0]:.0f} – {T_range[1]:.0f} K")
else:
    print("  None found — all points have |Δ| ≤ 0.1")